In [ ]:
import os

# Os par?metros do job t?m nomes curtos; boto3 l? as vari?veis AWS abaixo.
PARAMETROS = {
    "S3_ENDPOINT": "S3_ENDPOINT",
    "AWS_DEFAULT_REGION": "S3_REGION",
    "AWS_ACCESS_KEY_ID": "ACCESS_KEY",
    "AWS_SECRET_ACCESS_KEY": "SECRET_KEY",
}
for variavel, parametro in PARAMETROS.items():
    dbutils.widgets.text(parametro, "")
    valor = dbutils.widgets.get(parametro) or os.environ.get(variavel)
    if not valor:
        raise ValueError(f"Par?metro obrigat?rio ausente: {parametro}")
    os.environ[variavel] = valor

S3_BUCKET = "DataLakeVetice"
RAW = "/Volumes/lakehouse_vertice/bronze/raw"
print("ambiente pronto")


In [0]:
import boto3
from botocore.config import Config

# sessão nova para não reusar credenciais em cache de sessões antigas
s3 = boto3.Session().client(
    "s3",
    endpoint_url=os.environ["S3_ENDPOINT"],
    config=Config(s3={"addressing_style": "path"}),   # o Supabase só entende o bucket no caminho da URL
)

for bucket in s3.list_buckets()["Buckets"]:
    print(bucket["Name"])

DataLakeVetice


In [0]:
objetos = []
for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket=S3_BUCKET):
    objetos += pagina.get("Contents", [])

# só arquivos (o storage cria um objeto vazio de marcação em cada pasta)
arquivos = [o["Key"] for o in objetos if not o["Key"].endswith("/") and o["Size"] > 0
            and not o["Key"].endswith(".emptyFolderPlaceholder")]

from collections import Counter
for pasta, n in sorted(Counter(k.split("/")[0] for k in arquivos).items()):
    print(f"{pasta:<12} {n:>3} arquivos")
print(f"{'total':<12} {len(arquivos):>3}")

clickup-20261020T223931Z-1-001  25 arquivos
compras-20261020T223900Z-1-001   1 arquivos
documentos-20261020T223900Z-1-001   9 arquivos
excel-20261020T223859Z-1-001  20 arquivos
financeiro-20261020T223857Z-1-001   1 arquivos
msproject-20261020T223856Z-1-001  15 arquivos
pmo-20261020T223854Z-1-001   2 arquivos
rh-20261020T223853Z-1-001   1 arquivos
timesheet-20261020T223855Z-1-001   1 arquivos
total         75


In [0]:
TABELAS = ["projetos", "status_semanal", "excel_tarefas", "clickup_tarefas", "msproject_tarefas",
           "colaboradores", "apontamentos", "custos", "entregas_fornecedor"]

for tabela in TABELAS:
    spark.sql(f"DROP TABLE IF EXISTS lakehouse_vertice.bronze.{tabela}")

for pasta in sorted({k.split("/")[0] for k in arquivos}):
    dbutils.fs.rm(f"{RAW}/{pasta}", True)
    print("apagada:", f"{RAW}/{pasta}")

apagada: /Volumes/lakehouse_vertice/bronze/raw/clickup-20261020T223931Z-1-001
apagada: /Volumes/lakehouse_vertice/bronze/raw/compras-20261020T223900Z-1-001
apagada: /Volumes/lakehouse_vertice/bronze/raw/documentos-20261020T223900Z-1-001
apagada: /Volumes/lakehouse_vertice/bronze/raw/excel-20261020T223859Z-1-001
apagada: /Volumes/lakehouse_vertice/bronze/raw/financeiro-20261020T223857Z-1-001
apagada: /Volumes/lakehouse_vertice/bronze/raw/msproject-20261020T223856Z-1-001
apagada: /Volumes/lakehouse_vertice/bronze/raw/pmo-20261020T223854Z-1-001
apagada: /Volumes/lakehouse_vertice/bronze/raw/rh-20261020T223853Z-1-001
apagada: /Volumes/lakehouse_vertice/bronze/raw/timesheet-20261020T223855Z-1-001


In [0]:
import os

def baixar(key):
    destino = f"{RAW}/{key}"                                   # o mesmo caminho do bucket
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    conteudo = s3.get_object(Bucket=S3_BUCKET, Key=key)["Body"].read()   # 1 · extrair
    with open(destino, "wb") as f:                                        # 2 · pousar
        f.write(conteudo)
    return len(conteudo)

total = 0
for key in arquivos:
    total += baixar(key)
print(f"{len(arquivos)} arquivos, {total/1e6:.1f} MB no raw")

75 arquivos, 6.9 MB no raw


In [0]:
%sql
-- CSVs do raw → bronze
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.projetos AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/pmo-*/pmo/projetos.csv',
                format => 'csv', header => true, inferColumnTypes => false);

CREATE OR REPLACE TABLE lakehouse_vertice.bronze.status_semanal AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/pmo-*/pmo/status_semanal.csv',
                format => 'csv', header => true, inferColumnTypes => false);

CREATE OR REPLACE TABLE lakehouse_vertice.bronze.colaboradores AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/rh-*/rh/colaboradores.csv',
                format => 'csv', header => true, inferColumnTypes => false);

CREATE OR REPLACE TABLE lakehouse_vertice.bronze.apontamentos AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/timesheet-*/timesheet/apontamentos.csv',
                format => 'csv', header => true, inferColumnTypes => false);

CREATE OR REPLACE TABLE lakehouse_vertice.bronze.custos AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/financeiro-*/financeiro/custos.csv',
                format => 'csv', header => true, inferColumnTypes => false);

CREATE OR REPLACE TABLE lakehouse_vertice.bronze.entregas_fornecedor AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/compras-*/compras/entregas_fornecedor.csv',
                format => 'csv', header => true, inferColumnTypes => false);

-- Excel / ClickUp / MS Project → bronze
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.excel_tarefas AS
SELECT `ID`               AS id,
       `Fase`             AS fase,
       `Tarefa`           AS tarefa,
       `Responsável`      AS responsavel,
       `Início previsto`  AS inicio_previsto,
       `Término previsto` AS termino_previsto,
       `Início real`      AS inicio_real,
       `Término real`     AS termino_real,
       `% Concluído`      AS pct_concluido,
       `Horas previstas`  AS horas_previstas,
       url_decode(_metadata.file_name) AS arquivo
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/excel-*/excel/',
                format => 'excel', headerRows => 1, inferColumnTypes => false);

CREATE OR REPLACE TABLE lakehouse_vertice.bronze.clickup_tarefas AS
SELECT list.id   AS list_id,
       list.name AS list_name,
       t.*,
       url_decode(_metadata.file_name) AS arquivo
FROM (
    SELECT list, explode(tasks) AS t, _metadata
    FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/clickup-*/clickup/',
                    format => 'json', multiLine => true)
);

CREATE OR REPLACE TABLE lakehouse_vertice.bronze.msproject_tarefas AS
SELECT * EXCEPT (_rescued_data),
       url_decode(_metadata.file_name) AS arquivo
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/msproject-*/msproject/',
                format => 'xml', rowTag => 'Task');

-- Conferência de volumes
-- Deve aparecer o mesmo de ontem:
-- apontamentos 30.883 · custos 23.640 · status_semanal 3.911 · clickup_tarefas 2.119 · excel_tarefas 1.663
-- msproject_tarefas 1.260 · entregas_fornecedor 640 · colaboradores 523 · projetos 60
SELECT 'projetos' AS tabela, COUNT(*) AS linhas FROM lakehouse_vertice.bronze.projetos
UNION ALL SELECT 'status_semanal', COUNT(*) FROM lakehouse_vertice.bronze.status_semanal
UNION ALL SELECT 'excel_tarefas', COUNT(*) FROM lakehouse_vertice.bronze.excel_tarefas
UNION ALL SELECT 'clickup_tarefas', COUNT(*) FROM lakehouse_vertice.bronze.clickup_tarefas
UNION ALL SELECT 'msproject_tarefas', COUNT(*) FROM lakehouse_vertice.bronze.msproject_tarefas
UNION ALL SELECT 'colaboradores', COUNT(*) FROM lakehouse_vertice.bronze.colaboradores
UNION ALL SELECT 'apontamentos', COUNT(*) FROM lakehouse_vertice.bronze.apontamentos
UNION ALL SELECT 'custos', COUNT(*) FROM lakehouse_vertice.bronze.custos
UNION ALL SELECT 'entregas_fornecedor', COUNT(*) FROM lakehouse_vertice.bronze.entregas_fornecedor
ORDER BY linhas DESC;

tabela,linhas
apontamentos,30883
custos,23640
status_semanal,3911
clickup_tarefas,2119
excel_tarefas,1663
msproject_tarefas,1260
entregas_fornecedor,640
colaboradores,523
projetos,60
